#Primary market data

In [0]:
%pip install yfinance

In [0]:
import yfinance as yf
import pandas as pd
from pyspark.sql import functions as F

In [0]:
tickers = {
    "^NSEI": "NIFTY_50",
    "^BSESN": "SENSEX",
    "^NSEBANK": "NIFTY_BANK",
    "^CNXENERGY": "NIFTY_ENERGY",
    "^CNXAUTO": "NIFTY_AUTO",
    "^CNXIT": "NIFTY_IT",
    "HAL.NS": "HAL",
    "INDIGO.NS": "INDIGO",
    "ASIANPAINT.NS": "ASIAN_PAINTS",
    "ONGC.NS": "ONGC",
    "BZ=F": "BRENT_CRUDE",
    "INR=X": "USD_INR",
    "GC=F": "GOLD",
    "^INDIAVIX": "INDIA_VIX"
}

In [0]:
dfs = []

for ticker, name in tickers.items():
    
    print(f"Downloading {ticker}")
    
    df = yf.download(
        ticker,
        start="2023-10-01",
        end="2025-04-01",
        interval="1d",
        progress=False
    )
    
    # ✅ CRITICAL FIX — flatten here
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    
    df = df.reset_index()
    df["ticker"] = ticker
    df["asset_name"] = name
    
    dfs.append(df)

In [0]:
final_df = pd.concat(dfs, ignore_index=True)

In [0]:
print(final_df.columns)


In [0]:
market_data= spark.createDataFrame(final_df)


In [0]:
market_data = market_data.select(
    F.col("Date").alias("trade_date"),
    F.col("Open").alias("open"),
    F.col("High").alias("high"),
    F.col("Low").alias("low"),
    F.col("Close").alias("close"),
    F.col("Volume").alias("volume"),
    "ticker",
    "asset_name"
).withColumn("ingestion_timestamp", F.current_timestamp()).withColumn("source_file", F.lit("yfinance_market_data"))

In [0]:
display(market_data)

In [0]:
# Save market_data to landing zone as Parquet
market_data.write.mode("overwrite").parquet("/Volumes/iran_israel_capstone_project/bronze/landing_zone/market_data/")



#Alpha Vantage-MACRO DATA


In [0]:
import requests
import pandas as pd
from pyspark.sql import functions as F
from datetime import datetime

# --- CONFIGURATION ---
API_KEY = "YOURC3ODNMPQM5TL20OM"  
BASE_URL = "https://www.alphavantage.co/query"
LANDING_PATH = "/Volumes/iran_israel_capstone_project/bronze/landing_zone/macro_data"

# Ensure landing directory exists
dbutils.fs.mkdirs(LANDING_PATH)

def fetch_macro_data(function, name, interval=None):
    """Fetches data from Alpha Vantage and saves to Bronze Landing Zone"""
    params = {
        "function": function,
        "apikey": API_KEY
    }
    if interval:
        params["interval"] = interval
    
    print(f"Fetching {name}...")
    response = requests.get(BASE_URL, params=params)
    data = response.json()
    
    # Extract the data list (Alpha Vantage typically returns a 'data' or 'values' key)
    # For CPI/WTI/BRENT, the key is usually "data"
    if "data" in data:
        df_pd = pd.DataFrame(data["data"])
        
        # Save as Parquet to Landing Zone
        spark_df = spark.createDataFrame(df_pd)
        
        # Add Audit Columns as required by your project doc 
        spark_df = spark_df.withColumn("ingestion_timestamp", F.current_timestamp()) \
                           .withColumn("source_name", F.lit(name)) \
                           .withColumn("source_file", F.lit(f"alpha_vantage_{function}"))
        
        target_path = f"{LANDING_PATH}/{name.lower()}"
        spark_df.write.mode("overwrite").parquet(target_path)
        print(f"Successfully saved {name} to {target_path}")
    else:
        print(f"Error fetching {name}: {data.get('Information', 'Unknown Error')}")

# Execute Ingestion for the 3 required data points [cite: 48]
fetch_macro_data("CPI", "India_CPI", interval="monthly")
fetch_macro_data("WTI", "WTI_Crude", interval="daily")
fetch_macro_data("BRENT", "Brent_Crude_Alpha", interval="daily")